主要教学内容
- 本实验通过构建一个“冷邮件营销自动化系统”（Cold Sales Outreach Email System），涵盖了 Agent 开发的三个核心进阶主题：
- Agent 工作流（Agent Workflow）：学习如何定义具备不同人设（专业、幽默、高效）的销售代理，并通过 Runner 编排它们的工作流程。
- 工具的使用（Use of Tools）：
  - 如何通过 @function_tool 装饰器将普通 Python 函数（如发送邮件的 send_email）直接封装为智能体可调用的工具。
  - 如何将一个 Agent 本身转换为另一个 Agent 的工具（as_tool），实现“智能体调用智能体”。
  - 智能体协作（Agent Collaboration）：
  - Handoffs（移交）：学习如何让一个智能体（如销售经理）在完成特定步骤后，将任务控制权移交给另一个专门的智能体（如邮件格式化/发送经理），这是实现复杂业务逻辑的关键。

教学目标

- 掌握 Agentic 设计模式：从简单的线性流程进阶到能够根据指令规划、生成、评价并执行复杂任务的智能体系统。
- 工程化自动化：通过集成第三方服务（如 SendGrid），让智能体具备真实的“行动力”，将生成的文本直接转化为实际的业务操作。
- 理解架构演进：理解“工作流（Workflow）”与“智能体（Agent）”在 Anthropic 等框架定义下的微妙差异，并练习如何通过 Handoffs 和工具嵌套搭建多智能体系统。

In [1]:
import sys
from pathlib import Path

try:
    # .py 脚本
    current_dir = Path(__file__).resolve().parent
except NameError:
    # .ipynb
    current_dir = Path.cwd()

project_root = current_dir.parent
sys.path.append(str(project_root))

import logging
from config.log_config import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

In [2]:
############### 效果类似 system prompt ##############

sale_instructions1 = """你是一名服务于 ComplAI 的销售代理。
ComplAI 是一家利用人工智能技术，提供用于确保 SOC2 合规性以及准备审计的 SaaS 工具的公司。
你负责撰写专业、严肃的冷启动推销邮件。"""

sale_instructions2 = """你是一名服务于 ComplAI 的销售代理。
ComplAI 是一家利用人工智能技术，提供用于确保 SOC2 合规性以及准备审计的 SaaS 工具的公司。
你负责撰写风趣幽默、引人入胜且更容易获得回复的冷启动推销邮件。"""

sale_instructions3 = """你是一名服务于 ComplAI 的销售代理。
ComplAI 是一家利用人工智能技术，提供用于确保 SOC2 合规性以及准备审计的 SaaS 工具的公司。
你负责撰写简明扼要、直奔主题的冷启动推销邮件。"""


$\color{red}{下面部分接口的 name 参数使用了中文, 可能会出错, 应该一律使用英文}$

各种`name, description, instruction`参数是 agent 能看到的核心属性, 会影响 agent 的决策, `description, instruction`可以用中文, `name`一般使用英文比较稳定

In [3]:
from openai import AsyncOpenAI
from agents import (
    Runner,
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)

import os

custom_client = AsyncOpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

set_default_openai_client(custom_client)

set_default_openai_api(
    "chat_completions"
)  # 强制使用 Chat Completions API 兼容阿里 DashScope

set_tracing_disabled(True)  # 禁用 Tracing Telemetry 避免 401 报错

In [21]:
from agents import Agent

sale_agent1 = Agent(
    name="专业的销售智能体", instructions=sale_instructions1, model="deepseek-v4-pro"
)

sale_agent2 = Agent(
    name="风趣引人的销售智能体",
    instructions=sale_instructions2,
    model="deepseek-v4-flash",
)

sale_agent3 = Agent(
    name="高效精炼的销售智能体", instructions=sale_instructions3, model="kimi-k2.6"
)

In [5]:
from openai.types.responses import (
    ResponseTextDeltaEvent,
    ResponseReasoningSummaryTextDeltaEvent,
)

In [6]:
############### 流式输出测试 ##############

# 并没有向 LLM 发起请求
# 不涉及网络等待，不需要 await
# 启动智能体执行任务, 返回的不是一个“最终结果”, 而是一个异步生成器(Async Generator), 类似一个流媒体播放器
result = Runner.run_streamed(starting_agent=sale_agent1, input="写一封冷启动邮件")

thinking, thinking_finished = False, False  # 非思考模型也不会仅打印一个空行

async for event in result.stream_events():
    if event.type == "raw_response_event":
        data = event.data
        if isinstance(data, ResponseReasoningSummaryTextDeltaEvent) and data.delta:
            if not thinking:
                thinking = True  # 标记正在思考
            print(
                f"\033[90m{data.delta}\033[0m", end="", flush=True
            )  # 淡灰色或括号包裹表示思考过程

        # 思考完毕，打印最终正文
        elif isinstance(data, ResponseTextDeltaEvent) and data.delta:
            if thinking and not thinking_finished:
                print("\n\n")
            print(data.delta, end="", flush=True)
            if not thinking_finished:
                thinking_finished = True

我们被要求写一封冷启动邮件（cold outreach email），作为销售代理，服务于 ComplAI。需要专业、严肃。所以邮件应该清晰、简洁，突出 ComplAI 的价值主张，针对 SOC2 合规和审计准备。称呼对方名字未知，可以用通用问候。主题行吸引注意力。内容：介绍 ComplAI，说明痛点（SOC2 合规耗时、复杂、人工），解决方案（AI 驱动），提供价值，呼吁行动，如安排演示。保持专业严肃，不要过于销售化。


**主题：简化 SOC2 合规流程，专注业务增长**

尊敬的[潜在客户姓名]，

您好，我是 ComplAI 的销售顾问。了解到贵公司正在经历或准备 SOC2 审计，我希望向您介绍我们如何帮助您显著降低合规工作的复杂性和时间成本。

SOC2 合规通常需要耗费大量人力去收集证据、管理控制措施和应对审计请求。ComplAI 利用人工智能技术将这些流程自动化，从实时监控控制点、自动生成证据，到审计就绪的完整报告，让您的团队从繁琐事务中解脱出来，同时确保无遗漏、无延误。

我们的客户中，已有超过 80% 在首次审计中将准备时间缩短了 50% 以上，并持续保持合规状态。我们提供 14 天免费试用，无需任何技术对接，即可体验智能合规管理带来的效率提升。

如果您本周有 15 分钟时间，我很乐意为您安排一次简短演示，展示 ComplAI 如何针对您的具体需求提供支持。您可以通过[链接/邮箱]直接预约时间。

期待与您进一步交流。

此致  
[你的姓名]  
销售顾问  
ComplAI  
[你的邮箱] | [你的电话] | ComplAI 官网

**Runner.run (同步/块级获取)**

- SDK 发送请求给 LLM，程序在此时进入阻塞等待（Blocking）状态
- LLM 完成所有思考、调用工具、生成完整回复后，一次性将结果打包返回
- 函数执行结束，返回一个完整的 AgentRunResult 对象
  
- 代码写起来最顺手，不需要处理流式循环
- 如果任务很重（比如需要调用 3 个工具、生成很长的报告），用户必须盯着空白屏幕等待整个过程全部跑完


**Runner.run_streamed (异步/实时获取)**

- 立即返回一个 RunResultStreaming 对象（这只是一个“开关”）
- 通过`async for event in result.stream_events():`循环订阅生成过程中的每一个片段
- 只要 LLM 吐出一个字（token），或者 Agent 做了一次工具调用，就能立刻在 async for 循环里收到对应的 event
  
- 实时反馈
- 可以实时捕获 Agent 的每一步“内心独白”（比如 tool_called 或 handoff_requested），而不仅仅是看到最后的文本

`result.stream_events()`

- 一个异步循环，用来从智能体那里获取实时产生的各种“事件”, 比如：
  - raw_response_event：模型生成的文本片段
  - tool_call_event：智能体决定调用工具了
  - agent_handoff_event：智能体决定把任务交给别人了

`if event.type == "raw_response_event" ...`

筛选器：代码只关心模型正在写的“正文内容”。它通过判断事件类型，过滤掉其他系统级的事件，只提取出真实的文本片段。

`ResponseTextDeltaEvent`，“Delta”指的是“增量”。也就是模型刚刚产生的这几个新字。

`end=""`防止`print`函数自动换行，保持文字连续。

`async for event in result.stream_events():...`

- 这行代码本身就是一个完整的异步任务执行单元
- 隐式 await：在 async for 的每一次迭代中，Python 会自动调用 __anext__()。如果当前网络还没收到下一个 token，这个 __anext__() 内部会发起一个真正的 await 操作，把控制权交还给事件循环


In [7]:
import asyncio

In [ ]:
############### 全量输出测试 ##############
#### jupyter notebook 会报错 ####

message = "写一封冷启动邮件"  # 类似 user prompt


async def write_emails():
    results = await asyncio.gather(  # 异步等待
        Runner.run(starting_agent=sale_agent1, input=message),
        Runner.run(starting_agent=sale_agent2, input=message),
        Runner.run(starting_agent=sale_agent3, input=message),
    )
    return results


results = asyncio.run(write_emails())

In [9]:
#### jupyter notebook 流式输出替代 ####
message = "写一封冷启动邮件"  # 类似 user prompt
results = await asyncio.gather(  # 异步等待
    Runner.run(starting_agent=sale_agent1, input=message),
    Runner.run(starting_agent=sale_agent2, input=message),
    Runner.run(starting_agent=sale_agent3, input=message),
)

In [10]:
outputs = [result.final_output for result in results]
print(*outputs, sep="\n\n************************\n\n")

**主题：简化您的 SOC 2 合规流程，让审计准备不再焦虑**

尊敬的 [收件人姓名]，您好：

SOC 2 合规往往意味着无尽的文档整理、证据收集和跨部门沟通，而审计准备阶段的压力更是让团队疲于奔命。我们深知，对于追求卓越的安全与合规团队而言，效率与精确性同样重要。

ComplAI 是一家专注于利用人工智能技术，帮助企业实现 SOC 2 合规自动化的 SaaS 平台。我们不仅将证据采集和策略映射的效率提升了数倍，还能在审计前提供实时差距分析，让您始终清楚自己的合规状态，而非等到最后一刻才匆忙应对。

通过 ComplAI，您可以：
- **自动映射控制措施**：AI 引擎持续分析您的技术栈，关联对应的信任服务标准，避免人工遗漏。
- **实时待办与证据中心**：跨部门协作任务一目了然，所有审计证据自动归集并保持最新。
- **审计就绪仪表盘**：随时导出符合审计要求的报告，将审计准备时间从数周缩短到数小时。

我们已帮助多家 SaaS 公司以更低的成本和更少的人力投入，顺利通过并维持 SOC 2 合规。如果您希望了解 ComplAI 如何为您的下一轮审计做好万全准备，我很乐意安排一次 15 分钟的简短演示，向您展示实际效果。

您是否方便在下周初简单交流？期待您的回复。

顺祝商祺，

[您的姓名]  
[职位] | ComplAI  
[联系电话]  
[公司网站]

************************

**主题行**：您的 SOC2 审计，AI 帮您“躺平”搞定  

嘿 [姓名]，  

如果“准备 SOC2 审计”是一项奥运项目，您可能已经拿了十块金牌——但谁不想省下那点喘气的时间呢？  

我是 ComplAI 的 [你的名字]。我们做了个 AI 工具，专门帮您把合规变成“自动打勾机”——  
- 自动映射控制到证据  
- 实时监控差距  
- 生成审计就绪报告，仿佛您有个 24/7 的虚拟合规官（它不喝咖啡，也不抱怨加班）  

最棒的是：  
**您只需像往常一样工作，AI 会默默替您把审计师最爱的那些“小细节”处理好。**  

上周有个客户说，用了 ComplAI 后，他们首次审计准备时间从 3 个月砍到 2 周——而且全程没掉头发。  

要不咱们约个 10 分钟的“电子咖啡”？我给您演示一下：  
让您以后面对审计师时

In [11]:
sale_picker_instruction = """从给出的选项中选择最佳的冷启动邮件,
想象你是一名顾客, 选择一封你最想回复的邮件,
不需要解释原因, 直接回复即可
"""

sales_picker_agent = Agent(
    name="邮件筛选员", instructions=sale_picker_instruction, model="qwen3.7-plus"
)

In [12]:
emails = "冷启动推销邮件: " + "\n\nEmail:\n\n".join(outputs)

best = await Runner.run(starting_agent=sales_picker_agent, input=emails)  #

print(f"最佳邮件:\n{best.final_output}")

最佳邮件:
第三封邮件（主题：将 SOC2 合规准备时间从数月缩短至数周）


Jupyter Notebook 从 IPython 7.0 版本开始，默认启用了 Top-Level await 功能。它的工作原理如下：

后台始终运行着一个事件循环（Event Loop）： Jupyter 在启动你的内核（Kernel）时，后台就已经跑着一个 asyncio 的事件循环了

自动代码包装： 当你在 Notebook 的单元格（Cell）里写了顶级的 await 时，Jupyter 在执行该单元格前，会在后台**隐式（悄悄地）**把这个单元格的代码包裹进一个异步函数中，然后将其提交给后台的事件循环去执行。

In [13]:
############### 接下来是工具调用 ##############
from agents import function_tool

In [14]:
# 作用是直接封装为foundations 那一章节的类似 record_user_details_json 的格式
@function_tool
def send_email(email_body: str):
    logger.info(f"send email:{email_body[:30]}")

In [15]:
description = "写一封冷启动邮件"

sale_agent_tool1 = sale_agent1.as_tool(
    tool_name="sale_agent_tool1", tool_description=description
)
sale_agent_tool2 = sale_agent2.as_tool(
    tool_name="sale_agent_tool2", tool_description=description
)
sale_agent_tool3 = sale_agent3.as_tool(
    tool_name="sale_agent_tool3", tool_description=description
)

sale_tools = [sale_agent_tool1, sale_agent_tool2, sale_agent_tool3, send_email]

In [16]:
sale_manager_instruction = """你是一名 ComplAI 公司的销售经理。你的目标是通过使用 sale_agent_tool，筛选出一封最出色的销售冷启动邮件。请务必严格按照以下步骤执行：
生成草稿：使用所有三个 sale_agent_tool 分别生成三份不同的邮件草稿。在三份草稿全部就绪之前，请勿进入下一步。
评估与挑选：审阅这三份草稿，根据你的判断，选出其中转化效果最好的一封。
发送邮件：使用 send_email 工具发送你选出的那封（且仅发送这一封）最优邮件。

核心准则：
你必须使用 sale_agent_tool 来生成草稿 —— 禁止自行撰写。
你必须使用 send_email 工具发送且仅发送 一封 邮件 —— 绝不能多于一封。”
"""

sale_manager_agent = Agent(
    name="销售经理智能体",
    instructions=sale_manager_instruction,
    tools=sale_tools,  # type: ignore
    model="deepseek-v4-pro",
)

await Runner.run(starting_agent=sale_manager_agent, input="给亲爱的ceo发送一封邮件")

2026-06-28 09:20:35,550 [INFO] __main__: send email:**主题：将SOC2合规时间从数月缩短至数周——无需增加合规


RunResult(input='给亲爱的ceo发送一封邮件', new_items=[ReasoningItem(agent=Agent(name='销售经理智能体', handoff_description=None, tools=[FunctionTool(name='sale_agent_tool1', description='写一封冷启动邮件', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7ea4ca9b8680>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None), FunctionTool(name='sale_agent_tool2', description='写一封冷启动邮件', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': 

In [17]:
subject_instruction = """你可以为冷启动销售邮件撰写邮件主题。你将收到一段信息，你的任务是写出一个能够最大程度吸引收件人回复的邮件主题。"""
html_instruction = """你可以将文本格式的邮件正文转换为 HTML 格式。
你将收到一段可能包含 Markdown 标记的文本邮件正文，你的任务是将其转换为布局设计简洁、清晰且具有感染力的 HTML 格式邮件。"""

subject_agent = Agent(
    name="主题撰写智能体", instructions=subject_instruction, model="kimi-k2.6"
)
html_agent = Agent(
    name="格式转换智能体", instructions=html_instruction, model="qwen3.7-max"
)

subject_writer_tool = subject_agent.as_tool(
    tool_name="邮件主题撰写工具",
    tool_description="为一封冷启动推销邮件撰写主题(subject)",
)
html_converter_tool = subject_agent.as_tool(
    tool_name="html 转换工具", tool_description="将邮件的正文内容转换为 html 格式"
)


@function_tool
def send_html_email(email_body: str):
    logger.info(f"send html email:{email_body[:30]}")


emailer_tools = [subject_writer_tool, html_converter_tool, send_html_email]

In [22]:
emailer_instruction = """首先，使用"邮件主题撰写工具"为该邮件撰写一个主题。
接着，使用 "html 转换工具" 将邮件正文转换为 HTML 格式。
最后，使用 send_html_email 工具，将处理好的邮件主题和 HTML 正文发送出去。”
"""

# handoff_description 类似 api doc
# 虽然后面的 sale_manager_instruction 中也提到了, 但是  sale_manager_instruction 是给 sale_manager_agent 看的
# 而 handoff_description 是 emailer_agent 自己的标签
emailer_agent = Agent(
    name="email manager agent",
    instructions=emailer_instruction,
    tools=emailer_tools,
    model="deepseek-v4-pro",
    handoff_description="将邮件转换为 html 格式并发送",
)  # 并没有作为工具

In [23]:
# 与 tool 的功能类似, 但是这是以 agent 的形式传递给 sale_manager_agent
# 而 tool 是 agent 又包装了一层
handoffs = [emailer_agent]

In [24]:
sale_manager_instruction = """你是一名 ComplAI 公司的销售经理。你的目标是通过使用 sale_agent_tool，筛选出一封最出色的销售冷启动邮件。请务必严格按照以下步骤执行：
生成草稿：使用所有三个 sale_agent_tool 分别生成三份不同的邮件草稿。在三份草稿全部就绪之前，请勿进入下一步。
评估与挑选：审阅这些草稿，根据你的判断，选出其中最好的一封。如果你对初次生成的结果不满意，可以多次调用工具进行修改。
移交发送：仅将获胜的那份邮件草稿移交给"email manager agent"。由"email manager agent"负责后续的排版和发送工作。

核心准则：
你必须使用 sale_agent 工具来生成草稿 —— 禁止自行撰写。
你必须恰好移交 一封 邮件给“邮件管理智能体” —— 绝不能多于一封。
"""

sale_manager_agent = Agent(
    name="sale manager agent",
    instructions=sale_manager_instruction,
    tools=sale_tools,
    handoffs=handoffs,
    model="qwen3.7-plus",
)

await Runner.run(
    starting_agent=sale_manager_agent, input="发送一封冷启动邮件给亲爱的 ceo"
)

2026-06-28 09:33:31,882 [INFO] __main__: send html email:主题：SOC 2 in 3 weeks instead of


RunResult(input='发送一封冷启动邮件给亲爱的 ceo', new_items=[ReasoningItem(agent=Agent(name='sale manager agent', handoff_description=None, tools=[FunctionTool(name='sale_agent_tool1', description='写一封冷启动邮件', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7ea4ca9b8680>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None), FunctionTool(name='sale_agent_tool2', description='写一封冷启动邮件', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}